In [9]:
!pip install yfinance
!pip install requests
!pip install fredapi
!pip install python-dotenv

# 데이터 수집 노트북

이 노트북은 외부 소스(FRED, Yahoo Finance, CoinGecko)에서 원본 데이터를 받아 `data/raw/`에 CSV로 저장

- 기간: 2020-01-01 ~ 2023-05-31
- 출력: `data/raw/<name>.csv` (약 20~25개 파일)

## 수집 대상 목록

| 이벤트 | 필요 지표 |
|--------|----------|
| COVID-19 팬데믹 시작 (2020년 1월) | S&P 500, VIX, CPI |
| 연준 금리 인하 (2020년 3월) | 기준금리, 연준 자산 |
| 경제 봉쇄 및 경기 침체 | 실업률, GDP |
| 백신 개발 및 배포 (2020년 말) | 주식 지수, VIX |
| 인플레이션 상승 (2021년) | CPI, WTI 원유 |
| 공급망 문제 | 산업 생산 지수 |
| 암호화폐 붐 | BTC-USD, ETH-USD |
| 테슬라 및 전기차 | TSLA |
| 게임스톱 사태 (2021년 1월) | GME, AMC |
| 베드 배스 앤드 비욘드 파산 | BBBY |
| 코인베이스 상장 | COIN |
| 아크 인베스트먼트 | ARKK |

In [13]:
# Import & 환경 설정
import os
import sys
import time
sys.path.append('../')
from utils.data_loader import fetch_fred
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime

ModuleNotFoundError: No module named 'utils.data_loader'

In [12]:
# 공통 설정
START = '2020-01-01'
END = '2023-05-31'
RAW_DATA_PATH = '../data/raw/'
os.makedirs(RAW_DATA_PATH, exist_ok=True)
print(f'저장 경로: {os.path.abspath(RAW_DATA_PATH)}')
print(f'기간: {START} ~ {END}')

저장 경로: c:\Users\kimch\Desktop\project\pandemic-finance\data\raw
기간: 2020-01-01 ~ 2023-05-31


## 섹션 1. FRED 수집

FRED(Federal Reserve Economic Data)에서 거시경제 지표 10종을 수집

In [ ]:
# FRED 시리즈 일괄 수집
fred_series = {
    'SP500': 'SP500',
    'VIX': 'VIXCLS',
    'CPI': 'CPIAUCSL',
    'FedAssets': 'WALCL',
    'FedFundsRate': 'FEDFUNDS',
    'WTI': 'DCOILWTICO',
    '10Y2YSpread': 'T10Y2Y',
    'Unemployment': 'UNRATE',
    'GDP': 'GDP',
    'IndustrialProduction': 'INDPRO',
}
for name, series_id in fred_series.items():
    df = fetch_fred(series_id, START, END)
    df.to_csv(os.path.join(RAW_DATA_PATH, f'{name}.csv'), index=True)
    print(f'{name} ({series_id}) saved — rows={len(df)}')

In [ ]:
# FRED 수집 결과 확인 (head/tail/shape)
for name in fred_series:
    path = os.path.join(RAW_DATA_PATH, f'{name}.csv')
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    print(f'=== {name} === shape={df.shape}')
    print(df.head(2))
    print(df.tail(2))
    print()

## 섹션 2. Yahoo Finance 수집

`yfinance`로 주식/ETF/암호화폐 OHLCV 데이터 8종을 수집한다. 각 티커별 CSV에는 Open/High/Low/Close/Adj Close/Volume 컬럼이 포함됩니다.

In [ ]:
# yfinance 일괄 수집
tickers = ['GME', 'AMC', 'BBBY', 'TSLA', 'COIN', 'ARKK', 'BTC-USD', 'ETH-USD']
for ticker in tickers:
    df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=False)
    if df.empty:
        print(f'{ticker} — EMPTY (skipped)')
        continue
    df.to_csv(os.path.join(RAW_DATA_PATH, f'{ticker}.csv'), index=True)
    print(f'{ticker} saved — rows={len(df)}')

In [ ]:
# Yahoo Finance 수집 결과 확인
for ticker in tickers:
    path = os.path.join(RAW_DATA_PATH, f'{ticker}.csv')
    if not os.path.exists(path):
        
        print(f'{ticker} — FILE MISSING')
        continue
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    print(f'=== {ticker} === shape={df.shape}')
    print(df.head(2))
    print()

## 섹션 3. 수집 완료 검증

`data/raw/` 내 전체 CSV의 shape/컬럼/결측치 수를 출력하고, 빈 파일을 별도로 플래그합니다.

In [ ]:
# 모든 CSV 파일 목록 + 기본 통계 출력
files = sorted(f for f in os.listdir(RAW_DATA_PATH) if f.endswith('.csv'))
print(f'총 파일 수: {len(files)}')
for file in files:
    df = pd.read_csv(os.path.join(RAW_DATA_PATH, file))
    print(f'{file}: shape={df.shape}, columns={list(df.columns)}, nulls={int(df.isnull().sum().sum())}')

In [ ]:
# 결측치·빈 파일 체크 리포트
report = []
for file in files:
    df = pd.read_csv(os.path.join(RAW_DATA_PATH, file))
    if df.empty:
        report.append(f'[EMPTY]  {file}')
    else:
        null_count = int(df.isnull().sum().sum())
        tag = '[OK]   ' if null_count == 0 else '[NULLS]'
        report.append(f'{tag} {file} — nulls={null_count}, rows={len(df)}')
print('\n'.join(report))